# RegEx 

> Core regular expression patterns

In [ ]:
#| default_exp utils.re

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:

#| export
from __future__ import annotations
from typing import Dict, Optional, Any
import os, re, json
from pathlib import Path
from types import SimpleNamespace

## Regex Patterns
Central catalog of patterns reused across nbdev_mcp (aggregators, linting, docs parsing).

### Markdown Patterns

In [ ]:
#| export
HEADER_PATTERN = re.compile(r"^#{1,6}\s")
'''Match markdown headers (e.g., `# H1`, `## H2`).'''

REFERENCE_PATTERN = re.compile(r"^\s*\[([^\]]+)\]:\s*(.+)$")
'''Reference-style markdown links: `[id]: https://example.com`.'''

REF_LINK_USAGE_PATTERN = re.compile(r"\[([^\]]+)\]\[([^\]]+)\]")
'''Match markdown reference link usage: `[text][id]`.'''

BARE_REF_PATTERN = re.compile(r"(?<!\!)\[(?!\[)([^\]]+)\](?!\()")
'''Match bare reference link: `[id]` (not images, not followed by parens).'''

### Comment Patterns (TODOs, BUGs)

In [ ]:
#| export
TODO_CODE_PATTERN = re.compile(r"#\s*TODO:?\s*(.*)", re.IGNORECASE)
'''TODO comments inside code cells: `# TODO: fix this`.'''

TODO_MD_PATTERN = re.compile(r"(?:^|\s)TODO:?(.+)", re.IGNORECASE)
'''TODO call-outs inside markdown content.'''

BUG_CODE_PATTERN = re.compile(r"#\s*BUG:?\s*(.*)", re.IGNORECASE)
'''BUG comments inside code cells: `# BUG: known issue`.'''

BUG_MD_PATTERN = re.compile(r"(?:^|\s)BUG:?\s*(.*)", re.IGNORECASE)
'''BUG markers inside markdown cells.'''

BUG_PATTERN = re.compile(r"(?:#\s*)?BUG:?\s*(.*)", re.IGNORECASE)
'''General BUG pattern matching both code (`# BUG:`) and markdown (`BUG:`) formats.'''


### Import Patterns

In [ ]:
#| export
RELATIVE_IMPORT_PATTERN = re.compile(r"^\s*from\s+(\.+[\w\.]*)\s+import\s+(.+)$")
'''Detect relative imports so tooling can flag / rewrite them.'''

RELATIVE_LEVEL_PATTERN = re.compile(r"^(\.+)(.*)$")
'''Parse relative import levels (e.g., `..foo` -> level=2, module=foo).'''
# RELATIVE_IMPORT_PATTERN = re.compile(r"(\.+)(.*)$")

IMPORT_FROM_PATTERN = re.compile(r"^\s*from\s+([\w\.]+)\s+import\s+", re.MULTILINE)
'''Match `from module import ...` statements.'''

IMPORT_PATTERN = re.compile(r"^\s*import\s+([\w\.]+)", re.MULTILINE)
'''Match `import module` statements.'''

### nbdev Directive Patterns

In [ ]:
#| export
EXPORT_DIRECTIVE_PATTERN = re.compile(r"^\s*#\|\s*exporti?\b", re.MULTILINE)
'''Match nbdev export directives (`#| export`, `#| exporti`) in code cells.'''

EXPORTA_DIRECTIVE_PATTERN = re.compile(r"^\s*#\|\s*exporta\b", re.MULTILINE)
'''Match nbdev exporta directive (`#| exporta` = export all) in code cells.'''

EXPORT_ANY_RE = re.compile(r'^\s*#\|\s*export[ia]?\b', re.MULTILINE)
'''Combined pattern for any export directive (export, exporti, exporta).'''

DEFAULT_EXP_PATTERN = re.compile(r"#\|\s*default_exp\s+([\w\.]+)")
'''Capture nbdev default_exp module targets from notebooks.'''

HIDE_DIRECTIVE_PATTERN = re.compile(r"^\s*#\|\s*hide\b", re.MULTILINE)
'''Match nbdev hide directive (`#| hide`) in code cells.'''

EVAL_FALSE_PATTERN = re.compile(r"^\s*#\|\s*eval:\s*false\b", re.MULTILINE | re.IGNORECASE)
'''Match nbdev eval:false directive to skip cell execution during tests.'''


### Symbol Definition Patterns

In [ ]:
#| export
FUNCTION_DEF_PATTERN = re.compile(r"^\s*def\s+(\w+)\s*\(", re.MULTILINE)
'''Match function definitions and capture the function name.'''

ASYNC_FUNCTION_DEF_PATTERN = re.compile(r"^\s*async\s+def\s+(\w+)\s*\(", re.MULTILINE)
'''Match async function definitions and capture the function name.'''

CLASS_DEF_PATTERN = re.compile(r"^\s*class\s+(\w+)\s*[\(:]", re.MULTILINE)
'''Match class definitions and capture the class name.'''

ALL_DEFINITION_PATTERN = re.compile(r"^\s*__all__\s*=", re.MULTILINE)
'''Detect `__all__` definitions in modules.'''

### Test Patterns

In [ ]:
#| export
TEST_FUNCTION_PATTERN = re.compile(r"^\s*def\s+test_\w+\s*\(", re.MULTILINE)
'''Match test function definitions (def test_*).'''

ASSERT_PATTERN = re.compile(r"^\s*assert\s+", re.MULTILINE)
'''Match assert statements in code.'''

PYTEST_MARKER_PATTERN = re.compile(r"@pytest\.mark\.\w+", re.MULTILINE)
'''Match pytest markers (@pytest.mark.*).'''

### Code Patterns

In [ ]:
#| export
MAIN_GUARD_PATTERN = re.compile(r"^\s*if\s+__name__\s*==\s*['\"]__main__['\"]\s*:", re.MULTILINE)
'''Detect `if __name__ == "__main__":` guards in code cells.'''


### File Mapping Patterns

In [ ]:
#| export
YAML_NAME_PATTERN = re.compile(r"^\s*name\s*:\s*([A-Za-z0-9._-]+)\s*$")
'''Extract environment name from conda/mamba YAML files.'''

NBDEV_CELL_COMMENT_PATTERN = re.compile(r"#\s*%%\s*(\.\./nbs/[\w/]+\.ipynb)\s+(\d+)")
'''Match nbdev cell comments in generated .py files: `# %% ../nbs/04_nb.ipynb 5`.'''

DIGIT_PREFIX = re.compile(r"^\d+_")
'''Match digit prefix e.g. `04_nb.ipynb`.'''

NBDEV_FILE_PREFIX = re.compile(r'^\d+[a-z]?_')
'''Match nbdev file prefix e.g. `04_nb.ipynb`.'''

### Catalog

In [ ]:
#| export
PATTERN_CATALOG: Dict[str, re.Pattern] = {
    # Markdown
    "header": HEADER_PATTERN,
    "reference_link": REFERENCE_PATTERN,
    "ref_link_usage": REF_LINK_USAGE_PATTERN,
    "bare_ref": BARE_REF_PATTERN,
    # Comments
    "todo_code": TODO_CODE_PATTERN,
    "todo_markdown": TODO_MD_PATTERN,
    "bug_code": BUG_CODE_PATTERN,
    "bug_markdown": BUG_MD_PATTERN,
    "bug": BUG_PATTERN,
    # Imports
    "relative_import": RELATIVE_IMPORT_PATTERN,
    "relative_level": RELATIVE_LEVEL_PATTERN,
    "import_from": IMPORT_FROM_PATTERN,
    "import": IMPORT_PATTERN,
    # nbdev directives
    "export_directive": EXPORT_DIRECTIVE_PATTERN,
    "exporta_directive": EXPORTA_DIRECTIVE_PATTERN,
    "export_any": EXPORT_ANY_RE,
    "default_exp": DEFAULT_EXP_PATTERN,
    "hide_directive": HIDE_DIRECTIVE_PATTERN,
    "eval_false": EVAL_FALSE_PATTERN,
    # Symbol definitions
    "function_def": FUNCTION_DEF_PATTERN,
    "async_function_def": ASYNC_FUNCTION_DEF_PATTERN,
    "class_def": CLASS_DEF_PATTERN,
    "all_definition": ALL_DEFINITION_PATTERN,
    # Tests
    "test_function": TEST_FUNCTION_PATTERN,
    "assert": ASSERT_PATTERN,
    "pytest_marker": PYTEST_MARKER_PATTERN,
    # Code
    "main_guard": MAIN_GUARD_PATTERN,
    # File mapping
    "yaml_name": YAML_NAME_PATTERN,
    "nbdev_cell_comment": NBDEV_CELL_COMMENT_PATTERN,
    "digit_prefix": DIGIT_PREFIX,
    "nbdev_file_prefix": NBDEV_FILE_PREFIX,
}
'''Lookup of named regular expressions used across the MCP codebase.'''

## Next

In [ ]:
#| export

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()